# Feature Engineering

Feature engineering is where the source datasets become model-ready `h3`/`date` tables. This section walks through the grid keys, raster-to-H3 aggregation, derived environmental variables, gradients, anomalies, environmental-regime labels, fishing features, and species-use features.


## H3 Grid and Analysis Keys

The analysis grid is defined from `config.yaml`, not from a data file. The configured Falkland Islands bounding box is expanded by `region.buffer_km` before H3 cells are generated. This buffer keeps nearby environmental and fishing context available around the study area rather than clipping exactly at the presentation boundary.

The grid is generated at the configured H3 resolution and written first with standard H3 string identifiers. A second grid file is then written with `h3` converted to unsigned 64-bit integers and sorted by that key. The integer form is used by most large tables because it is more compact and joins cleanly across Parquet, DuckDB, pandas, and GeoPandas workflows.

The H3 conversion is implemented as:

```python
gdf["h3"] = (
    gdf["h3"]
    .astype(str)
    .map(h3.str_to_int)
    .astype("uint64")
)
gdf = gdf.sort_values("h3").reset_index(drop=True)
```

Dynamic products use this spatial key together with a normalized daily date. Dates are converted to UTC, floored to the calendar day, and stored without timezone information. This avoids mismatches between timezone-aware and timezone-naive timestamps when reading filtered Parquet partitions.

The date normalization is implemented as:

```python
out["date"] = (
    pd.to_datetime(out["date"], utc=True)
    .dt.normalize()
    .dt.tz_localize(None)
)
```

Most dynamic tables use `(h3, date)` as the join key. Species-expanded products add `species` as a third key.


## Raster-to-H3 Lookups

Environmental rasters and bathymetry are converted to the H3 grid through reusable pixel-to-H3 lookup tables. For each configured raster dataset with `build_lookup: true`, the lookup builder:

1. opens one reference raster from `data/raw/<dataset>/`,
2. converts each raster pixel to a longitude-latitude polygon,
3. intersects candidate pixels with each H3 cell polygon, and
4. stores the geodesic overlap area and normalized H3-cell weight.

For H3 cell $i$ and raster pixel $p$, the stored weight is:

$$
w(i,p)=\frac{a(i,p)}{\sum_{q \in P(i)} a(i,q)}
$$

where $a(i,p)$ is the geodesic area of the pixel-H3 overlap. During daily feature construction, invalid raster values are dropped and valid pixels are renormalized by their remaining weights:

$$
X(i,t)=\frac{\sum_{p \in P_v(i,t)} w(i,p)X(p,t)}{\sum_{p \in P_v(i,t)} w(i,p)}
$$

The lookup files are written to `data/processed/*_lookup.parquet` and reused by environmental feature builders. This keeps raster aggregation deterministic across products and avoids recalculating geometric intersections for every date.


## Derived Environmental Variables

Wind speed:

$$
W(h,t)=\sqrt{u_{10}(h,t)^2+v_{10}(h,t)^2}
$$

Log chlorophyll-a:

$$
C_{log}(h,t)=\log(1+C(h,t))
$$

Seasonal encodings use adjusted day of year $d^*$ on a 365-day cycle:

$$
S_d=\sin\left(\frac{2\pi d^*}{365}\right), \quad C_d=\cos\left(\frac{2\pi d^*}{365}\right)
$$

## Seasonal Lookup

Seasonal timing is represented with an adjusted day-of-year on a 365-day cycle. Leap years are corrected by subtracting one day after February 28, so seasonal features remain comparable across years.

The reusable lookup table is written to `data/processed/seasonal_lookup.parquet` with one row per adjusted day of year:

- `adjusted_doy`: integer day on the 1-365 seasonal cycle.
- `doy_sin`: sine encoding of the seasonal cycle.
- `doy_cos`: cosine encoding of the seasonal cycle.

The same cyclic encoding is applied when derived environmental features are added to yearly feature partitions:

$$
S_d=\sin\left(\frac{2\pi d^*}{365}\right), \quad C_d=\cos\left(\frac{2\pi d^*}{365}\right)
$$

where $d^*$ is the leap-corrected adjusted day of year. These cyclic variables let models learn seasonal structure without treating January 1 and December 31 as far apart.


## H3 Neighbor Gradients

H3 ring-1 neighbor relationships are precomputed and reused. For each date, each focal cell is compared with valid neighbors:

$$
G_X(i,t)=\sqrt{\mathrm{mean}_{j \in N(i)}((X(i,t)-X(j,t))^2)}
$$

Gradients are calculated for SST, log-transformed chlorophyll-a, and SSH. These features represent local environmental contrast.

## Anomalies and Climatology

Temporal anomalies are computed relative to local seasonal conditions. For each H3 cell and adjusted day of year, a climatological mean is estimated across the environmental record. Daily anomalies are then:

$$
A_X(i,t)=X(i,t)-\bar{X}_i(d^*)
$$

Anomalies are calculated for SST, log-transformed chlorophyll-a, SSH, and wind speed.

## Environmental Regime Assignments

The workflow uses project-specific environmental regimes rather than treating an external seascape product as the final modeling layer. The NOAA/MBON 8-day seascapes were useful context, but they were not the selected operational representation because the rest of this project works on a daily `h3`/`date` grid and on the same environmental feature space used by the model. Building the regimes inside the workflow keeps the labels aligned with the Falkland-scale feature table, the yearly partitions, and the prediction products.

### SOM-Hierarchical Seascapes

The selected seascape pathway is the SOM-hierarchical classifier. It starts from the project feature grid, scales the environmental predictors, and trains a 15 x 15 self-organizing map. The SOM produces 225 prototype environmental states. Those prototype vectors are then clustered with Ward hierarchical clustering, and the active public workflow uses the k=30 cut.

Each `h3`/`date` row is assigned to its nearest SOM prototype and then inherits the hierarchical seascape class for that prototype. The assignment keeps three fields:

- `som_prototype`: nearest SOM prototype for the `h3`/`date` environmental state.
- `seascape`: selected hierarchical k=30 seascape class.
- `seascape_distance`: distance from the scaled feature vector to the assigned SOM prototype.

This gives the project a seascape label that is reproducible from the same features used downstream, instead of depending on a coarser external temporal product.

### Consolidated Environmental-Regime Table

Environmental regime labels are stored in `data/modeling/environmental_regimes/` as one `h3`/`date` table. This table is the active source for the selected SOM-hierarchical seascapes and for the Bayesian GMM environmental components.

The Bayesian GMM fields are:

- `bayesian_gmm_k30_component`: dominant environmental component.
- `bayesian_gmm_k30_component_probability`: probability of the dominant component.
- `bayesian_gmm_k30_component_entropy`: uncertainty across components.

Keeping these labels together avoids maintaining separate duplicate seascape assignment folders while still making both environmental groupings available to validation, maps, seascape-conditioned summaries, and diagnostic checks.


## Fishing Features

Fishing activity is represented as a fleet-concentration-weighted exposure index:

$$
F(h,t)=H(h,t)\times V(h,t)
$$

where $H$ is apparent fishing hours and $V$ is the number of unique vessels in the same cell-day.


## Species-Use Features

Species telemetry records are cleaned, converted to point geometries, spatially joined to the H3 grid, and aggregated by `h3`, `date`, and `species`. The species support fields are:

- `presence_count`: number of telemetry records in the cell-day-species group.
- `individual_count`: number of unique tagged individuals in the group.
- `trip_count`: number of unique trips in the group.

The current species-use target is the residence index:

$$
R(h,t,s)=P(h,t,s)\times I(h,t,s)
$$

where $P$ is `presence_count` and $I$ is `individual_count`.

For model training, observed `species`/`date` combinations are expanded across all H3 cells in the environmental feature grid. Cells without telemetry records for an observed species-date receive zero support and $R=0$; these are modeling zeros, not confirmed biological absences.


## Yearly Partitions

Most dynamic products are stored as yearly partitions using the pattern `year=<YYYY>/part.parquet`. This applies to environmental features, fishing effort features, species-presence features, model-ready training tables, predictions, and environmental-regime assignments.

The partitioning has three practical roles:

- it keeps large daily H3 tables small enough to inspect and rebuild one year at a time;
- it makes date-range operations explicit, because each partition belongs to one calendar year;
- it lets downstream scripts read only the years needed for a plot, diagnostic, model, or operational product.

Static products, such as the H3 grid, bathymetry/static features, neighbor tables, seasonal lookup, and raster-to-H3 lookup tables, are not partitioned by year because they do not change through time.


In [1]:
from pathlib import Path
import yaml

config = yaml.safe_load(Path("../config.yaml").read_text())
data_dir = Path("..") / config["paths"]["data"]
processed_dir = Path("..") / config["paths"]["processed"]

lookup_files = [
    processed_dir / f"{name}_lookup.parquet"
    for name, spec in config.get("datasets", {}).items()
    if spec.get("build_lookup", False)
]

artifacts = [
    *lookup_files,
    processed_dir / "h3_neighbors.parquet",
    processed_dir / "seasonal_lookup.parquet",
    data_dir / "features" / "static" / "static.parquet",
    data_dir / "features" / "environmental",
    data_dir / "features" / "fishing_effort",
    data_dir / "features" / "species_presence",
    data_dir / "modeling" / "environmental_regimes",
]

for artifact in artifacts:
    if artifact.is_dir():
        years = sorted(p.parent.name for p in artifact.glob("year=*/part.parquet"))
        detail = f"{len(years)} yearly partitions" if years else "directory present"
    else:
        detail = "present" if artifact.exists() else "missing"
    print(f"{artifact.relative_to(Path('..'))}: {detail}")


data/processed/sst_lookup.parquet: present
data/processed/chl_lookup.parquet: present
data/processed/ssh_lookup.parquet: present
data/processed/wind_lookup.parquet: present
data/processed/bathymetry_lookup.parquet: present
data/processed/h3_neighbors.parquet: present
data/processed/seasonal_lookup.parquet: present
data/features/static/static.parquet: present
data/features/environmental: 10 yearly partitions
data/features/fishing_effort: 10 yearly partitions
data/features/species_presence: 2 yearly partitions
data/modeling/environmental_regimes: 10 yearly partitions
